# Lab 09 — Use Cases: Generative Models in Real Contexts
*Casos de uso: modelos generativos en contexto real*

**Session 09 · Desarrollo y Evaluación de Modelos Propios** · Universidad del Rosario · Rosario GSB

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/09-real-world-use-cases/lab.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/09-real-world-use-cases/lab.ipynb)

In the slides we turned use cases into numbers: value × feasibility, KPIs and an ROI formula whose most important input is **q, the share of outputs that are usable without rework**. In this lab you build one small prototype per sector and measure its q, exactly as you would in the first week of a pilot.

**Learning objectives**
1. Compare a fine-tuned domain model (FinBERT) with a zero-shot LLM on the same finance task, including latency.
2. Build a retrieval prototype for symptom-based routing, measure top-3 accuracy and stress-test it (not medical advice).
3. Generate quiz questions with a small LLM and check automatically whether they are answerable.
4. Evaluate a sector agent with a knowledge base (Startti, or the Python alternative) on in-scope and out-of-scope questions.

**Time plan (60 min)**

| Part | Topic | Time |
|---|---|---|
| 1 | Finance: FinBERT vs a zero-shot LLM | 15' |
| 2 | Health: symptom → condition retrieval | 12' |
| 3 | Education: question generation + answerability check | 13' |
| 4 | Sector agent: Startti or Python RAG | 15' |
| — | Wrap-up | 5' |

**Kaggle checklist**
- *Settings → Internet* **on** (models and datasets are downloaded from Hugging Face).
- *Settings → Accelerator*: **GPU T4** makes Parts 1, 3 and 4 faster; CPU also works.
- Startti (optional, Part 4): *Add-ons → Secrets* → a secret named `STARTTI_API_KEY`, attached to this notebook. Without it, Part 4 uses the Python agent. Guide: [configuracion-startti.md](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-startti.md).

> ⚠️ **Not medical, financial or academic advice.** Every organization in this lab is fictional and every prototype is a teaching exercise.

**How to follow:** run each cell after the instructor (Shift + Enter). ✋ **Checkpoints** are answered in the Zoom chat; 🧪 **Try it** exercises are stretch goals with a hidden solution.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("datasets")
ensure("sentence-transformers", "sentence_transformers")
ensure("scikit-learn", "sklearn")

Shared imports, sizes and a helper that reads the course data files. `FAST_DEV_RUN` only shrinks the sizes (the course smoke test uses it); in class, leave it as is.

In [ ]:
import re, time, string, collections, urllib.request
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 110)

# Sizes: full lab vs. fast smoke test (same code path, smaller numbers)
N_FIN = 24 if FAST_DEV_RUN else 150        # financial tweets to score
N_SYM_KB = 300 if FAST_DEV_RUN else None   # symptom descriptions in the case library (None = all 853)
N_SYM_TEST = 40 if FAST_DEV_RUN else 212   # new descriptions to route
N_PAR = 2 if FAST_DEV_RUN else 8           # science paragraphs for question generation
MAX_NEW_TOKENS = 48 if FAST_DEV_RUN else 80

DATA_URL = "https://raw.githubusercontent.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/main/sessions/09-real-world-use-cases/data/"

def load_text(rel_path):
    """Read a course data file: the local copy first, then the course repository on GitHub."""
    local = Path("data") / rel_path
    if local.exists():
        return local.read_text(encoding="utf-8")
    with urllib.request.urlopen(DATA_URL + rel_path, timeout=30) as resp:
        return resp.read().decode("utf-8")

LAB_SUMMARY = {}  # the q of each prototype, collected for the wrap-up
print("Ready.")

## Part 1 — Finance: FinBERT vs a zero-shot LLM (15')

**Scenario.** *Andes Bank* (a fictional digital bank) prepares a morning brief for its advisors. Thousands of English headlines arrive overnight; the team wants to read the **bearish** and **bullish** ones first. Which model should sort them?

- **FinBERT** (`ProsusAI/finbert`; Araci, 2019): BERT further trained and fine-tuned on financial text. Small, fast, one job.
- **A zero-shot LLM** (`Qwen/Qwen2.5-0.5B-Instruct`): no training; we score each label word by its log-probability, as in S08.

Dataset: `zeroshot/twitter-financial-news-sentiment`, labels **0 = Bearish, 1 = Bullish, 2 = Neutral**.

In [ ]:
fin = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
fin_sample = fin.shuffle(seed=SEED).select(range(N_FIN))
fin_texts, fin_y = list(fin_sample["text"]), list(fin_sample["label"])
FIN_LABELS = ["bearish", "bullish", "neutral"]  # index = dataset label id
print(f"{len(fin_texts)} tweets | label counts:", collections.Counter(FIN_LABELS[y] for y in fin_y))
print("Example:", fin_texts[0], "→", FIN_LABELS[fin_y[0]])

⚠️ **Label mapping.** FinBERT and the dataset use different names *and* different ids. FinBERT says `positive / negative / neutral` (ids 0/1/2); the dataset says `bearish / bullish / neutral` (ids 0/1/2). Matching by id would silently swap bullish and bearish. We map by **name**: positive → bullish, negative → bearish.

In [ ]:
finbert = pipeline("text-classification", model="ProsusAI/finbert", device=DEVICE)
print("FinBERT labels:", finbert.model.config.id2label)
FINBERT_TO_DATASET = {"negative": 0, "positive": 1, "neutral": 2}   # by name, not by id

t0 = time.perf_counter()
finbert_out = finbert(fin_texts, batch_size=16, truncation=True)
finbert_ms = 1000 * (time.perf_counter() - t0) / len(fin_texts)
finbert_pred = [FINBERT_TO_DATASET[o["label"]] for o in finbert_out]
print(f"FinBERT accuracy: {accuracy_score(fin_y, finbert_pred):.3f} | {finbert_ms:.1f} ms per tweet")

Now the zero-shot LLM. For each tweet we build a chat prompt and compute $\log p(\text{label tokens}\mid\text{prompt})$ for each label word; the highest score wins. We load the LLM once and reuse it in Parts 3 and 4.

In [ ]:
LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
# .float(): recent transformers versions load this model in bfloat16, which is slower and less stable on a CPU
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME).float().to(DEVICE).eval()

def label_logprobs(text, verbalizer):
    """Sum of log-probabilities of each label word as the assistant's answer."""
    messages = [
        {"role": "system", "content": "You are a financial news analyst."},
        {"role": "user", "content": "Classify the market sentiment of this financial tweet for an investor.\n\n"
                                    f"Tweet: {text}\n\nAnswer with one word: {', '.join(verbalizer)}."},
    ]
    enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    prompt_ids = enc["input_ids"][0]
    scores = []
    for word in verbalizer:
        word_ids = tokenizer(word, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
        ids = torch.cat([prompt_ids, word_ids]).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = llm(input_ids=ids).logits[0]
        logp = torch.log_softmax(logits[len(prompt_ids) - 1:-1].float(), dim=-1)
        scores.append(logp.gather(1, word_ids.to(DEVICE).unsqueeze(1)).sum().item())
    return scores

def llm_classify(texts, verbalizer=FIN_LABELS):
    return [int(np.argmax(label_logprobs(t, verbalizer))) for t in texts]

t0 = time.perf_counter()
llm_pred = llm_classify(fin_texts)
llm_ms = 1000 * (time.perf_counter() - t0) / len(fin_texts)
print(f"Zero-shot LLM accuracy: {accuracy_score(fin_y, llm_pred):.3f} | {llm_ms:.1f} ms per tweet")

Side by side: accuracy, macro-F1 (every class counts the same, which matters because *neutral* dominates) and latency. The first row is a **baseline without any model**: always answer the most frequent label. A model that does not beat it on macro-F1 adds nothing.

In [ ]:
def fin_report(name, pred, ms):
    return {"model": name, "accuracy": round(accuracy_score(fin_y, pred), 3),
            "macro_F1": round(f1_score(fin_y, pred, average="macro"), 3), "ms_per_tweet": round(ms, 1)}

majority = collections.Counter(fin_y).most_common(1)[0][0]   # a model-free baseline: always the most frequent label
fin_table = pd.DataFrame([fin_report(f"Baseline: always '{FIN_LABELS[majority]}'", [majority] * len(fin_y), 0.0),
                          fin_report("FinBERT (fine-tuned)", finbert_pred, finbert_ms),
                          fin_report("Qwen2.5-0.5B zero-shot", llm_pred, llm_ms)])
display(fin_table)

for name, pred in [("FinBERT", finbert_pred), ("Zero-shot LLM", llm_pred)]:
    cm = pd.crosstab(pd.Series([FIN_LABELS[y] for y in fin_y], name="true"),
                     pd.Series([FIN_LABELS[p] for p in pred], name=f"{name} predicted"))
    display(cm)

LAB_SUMMARY["finance · best model accuracy"] = float(fin_table["accuracy"].iloc[1:].max())

✋ **Checkpoint (Zoom chat).** Does the zero-shot LLM beat the baseline? Look at its confusion matrix: which label does it predict almost always? Small LLMs often have a strong prior for one label word (*label bias*), so their zero-shot scores need calibration or examples before they are useful. For the morning brief, which error is costlier: a **bearish** tweet predicted as *neutral* (the advisor reads it late) or a *neutral* tweet predicted as bearish (the advisor reads it for nothing)? Which model makes more of the costly error in your confusion matrices?

💼 **Business lens.** Latency and cost matter too: FinBERT (~110M parameters) classifies in milliseconds on a CPU; the LLM needs three forward passes per tweet. If both had the same accuracy, which would you run on 60,000 headlines a month?

🧪 **Try it.** The words we use for the labels (the *verbalizer*) change the zero-shot result. Re-run the LLM with `["negative", "positive", "neutral"]` (same order as the dataset ids) and compare.

<details><summary>Show a solution</summary>

```python
alt_pred = llm_classify(fin_texts, verbalizer=["negative", "positive", "neutral"])
print(fin_report("Qwen · negative/positive/neutral", alt_pred, llm_ms))
```
The order must still follow the dataset ids (0 bearish, 1 bullish, 2 neutral). Differences of several points between verbalizers are common with small models: one more reason to measure on your own data before choosing.
</details>

## Part 2 — Health: symptom → condition retrieval (12')

> ⚠️ **Not medical advice.** This prototype only suggests which *case type* a request resembles, so that a nurse can route it faster. A health professional always decides. The data are short English descriptions from a public dataset, not real patients.

**Scenario.** *Clínica Serena* (fictional) receives appointment requests where patients describe symptoms in their own words. A nurse reads each one and picks the right schedule. Idea: show the nurse the **3 most similar past case types**.

**Approach (retrieval, no training):** embed a library of labelled descriptions with `sentence-transformers/all-MiniLM-L6-v2`; for a new description, find the *k* nearest ones and let them vote. The KPI is **top-3 accuracy**: is the right condition among the three suggestions?

Dataset: `gretelai/symptom_to_diagnosis` (`input_text` → `output_text`, 22 conditions).

In [ ]:
sym = load_dataset("gretelai/symptom_to_diagnosis")
kb = sym["train"] if N_SYM_KB is None else sym["train"].shuffle(seed=SEED).select(range(N_SYM_KB))
test = sym["test"].shuffle(seed=SEED).select(range(N_SYM_TEST))
kb_texts, kb_labels = list(kb["input_text"]), list(kb["output_text"])
test_texts, test_labels = list(test["input_text"]), list(test["output_text"])
print(f"Case library: {len(kb_texts)} descriptions · {len(set(kb_labels))} conditions | new requests: {len(test_texts)}")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
kb_vecs = embedder.encode(kb_texts, normalize_embeddings=True, batch_size=64)

def suggest(texts, k=5, n=3):
    """Top-n conditions for each text, by similarity-weighted vote of its k nearest library cases."""
    q = embedder.encode(texts, normalize_embeddings=True, batch_size=64)
    sims = q @ kb_vecs.T
    out = []
    for row in sims:
        votes = collections.defaultdict(float)
        for i in np.argsort(-row)[:k]:
            votes[kb_labels[i]] += float(row[i])
        out.append([c for c, _ in sorted(votes.items(), key=lambda kv: -kv[1])][:n])
    return out

def top_k_accuracy(suggestions, labels, n):
    return float(np.mean([y in s[:n] for s, y in zip(suggestions, labels)]))

sugg = suggest(test_texts, k=5)
print(f"top-1 accuracy: {top_k_accuracy(sugg, test_labels, 1):.3f} | top-3 accuracy: {top_k_accuracy(sugg, test_labels, 3):.3f}")
print("\nExample request:", test_texts[0])
print("Suggested case types (for the nurse, not a diagnosis):", sugg[0], "| true:", test_labels[0])

How much do the **labelled examples** add? A baseline with no examples: compare each request only with the 22 condition *names*.

In [ ]:
conditions = sorted(set(kb_labels))
cond_vecs = embedder.encode(conditions, normalize_embeddings=True)
name_sims = embedder.encode(test_texts, normalize_embeddings=True) @ cond_vecs.T
name_sugg = [[conditions[j] for j in np.argsort(-row)[:3]] for row in name_sims]
print(f"Names only → top-1: {top_k_accuracy(name_sugg, test_labels, 1):.3f} | top-3: {top_k_accuracy(name_sugg, test_labels, 3):.3f}")

Real requests are messier than a curated dataset. A quick **stress test**: keep only the first sentence (short mobile messages) or add typing errors.

In [ ]:
def first_sentence(text):
    return re.split(r"(?<=[.!?])\s+", text.strip())[0]

def add_typos(text, rate=0.15, seed=SEED):
    """Swap or drop one letter in about `rate` of the longer words (deterministic)."""
    r = random.Random(seed)
    words = []
    for w in text.split():
        if len(w) > 3 and r.random() < rate:
            i = r.randrange(1, len(w) - 1)
            w = w[:i] + w[i + 1:] if r.random() < 0.5 else w[:i] + w[i + 1] + w[i] + w[i + 2:]
        words.append(w)
    return " ".join(words).lower()

stress = []
for name, fn in [("full text", lambda t: t), ("first sentence only", first_sentence), ("with typos", add_typos)]:
    s = suggest([fn(t) for t in test_texts], k=5)
    stress.append({"condition": name, "top1": round(top_k_accuracy(s, test_labels, 1), 3),
                   "top3": round(top_k_accuracy(s, test_labels, 3), 3)})
display(pd.DataFrame(stress))
LAB_SUMMARY["health · top-3 accuracy (full text)"] = stress[0]["top3"]

✋ **Checkpoint.** Which number would you report to the clinic, top-1 or top-3, and why? What happens to the ROI of the case if real messages look like the "first sentence only" row?

💼 **Business lens.** The costliest error is not in this table: an **urgent** case routed to a slow schedule. A deployable version needs deterministic red-flag rules (chest pain → emergency line 123) *before* any model, and a KPI that tracks under-triage.

🧪 **Try it.** Change `k` (number of neighbours that vote) to 1, 3, 10 and 20. Which value maximizes top-3 accuracy here? Why can a large `k` hurt?

<details><summary>Show a solution</summary>

```python
for k in [1, 3, 5, 10, 20]:
    s = suggest(test_texts, k=k)
    print(k, round(top_k_accuracy(s, test_labels, 1), 3), round(top_k_accuracy(s, test_labels, 3), 3))
```
With `k = 1` there is only one candidate, so top-3 equals top-1. A moderate `k` lets similar cases vote; a very large `k` lets frequent but less similar conditions outvote the right one. Choose `k` on a validation split, never on the test set (S08!).
</details>

## Part 3 — Education: question generation + answerability check (13')

**Scenario.** A teacher at *Colibrí Online University* (fictional) wants practice questions generated from the course readings. A question that cannot be answered from the reading is worse than no question at all, so we check every generated question **automatically**:

1. **Grounded answer:** most words of the proposed answer appear in the paragraph.
2. **Round trip:** the model answers its own question from the paragraph; the new answer must match the proposed one (token F1 ≥ 0.5).

A question is **answerable** if it passes both. Data: `allenai/sciq`, which pairs science questions with the *support* paragraph they come from.

In [ ]:
sciq = load_dataset("allenai/sciq", split="validation")
sciq = sciq.filter(lambda r: len(r["support"].split()) >= 40)       # keep real paragraphs
paragraphs = sciq.shuffle(seed=SEED).select(range(N_PAR))
print(f"{len(sciq)} paragraphs with support text; using {len(paragraphs)}")

def chat(messages, max_new_tokens=MAX_NEW_TOKENS, prefix=""):
    """Greedy generation with the chat template (deterministic).
    `prefix` starts the assistant's answer for the model (e.g. "Question:") so it follows our format."""
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False) + prefix
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    return (prefix + tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)).strip()

LEVELS = {  # Bloom-style levels
    "remember": "a factual recall question (who, what, which or where)",
    "understand": "a question that asks why or how something happens",
}

def generate_question(paragraph, level="remember"):
    messages = [
        {"role": "system", "content": "You write short quiz questions for high-school science students."},
        {"role": "user", "content": f"Paragraph:\n{paragraph}\n\nWrite {LEVELS[level]} that can be answered using ONLY "
                                    "this paragraph, and its short answer (at most 6 words).\n"
                                    "Use exactly this format:\nQuestion: ...\nAnswer: ..."},
    ]
    raw = chat(messages, prefix="Question:")   # we write "Question:" for the model; it continues from there
    q = re.search(r"Question:\s*(.+)", raw)
    a = re.search(r"Answer:\s*(.+)", raw)
    return (q.group(1).strip() if q else ""), (a.group(1).strip() if a else "")

STOPWORDS = set("the a an of to in and or is are was were be by for with on at as it its this that "
                "which what who how why from into than then".split())

def content_tokens(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return [w for w in text.split() if w not in STOPWORDS]

def token_f1(pred, ref):
    p, r = content_tokens(pred), content_tokens(ref)
    common = sum((collections.Counter(p) & collections.Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def is_grounded(answer, paragraph, min_share=0.75):
    a, par = content_tokens(answer), set(content_tokens(paragraph))
    return bool(a) and sum(w in par for w in a) / len(a) >= min_share

def answer_from(paragraph, question):
    messages = [
        {"role": "system", "content": "Answer using only the paragraph, in at most 6 words. "
                                      "If the paragraph does not contain the answer, reply: not in the paragraph."},
        {"role": "user", "content": f"Paragraph:\n{paragraph}\n\nQuestion: {question}"},
    ]
    return chat(messages, max_new_tokens=16)

def check_questions(rows, level="remember"):
    records = []
    for row in rows:
        par = row["support"]
        q, a = generate_question(par, level)
        re_answer = answer_from(par, q) if q else ""
        grounded, f1 = is_grounded(a, par), token_f1(re_answer, a)
        records.append({"question": q, "answer": a, "re_answer": re_answer, "grounded": grounded,
                        "round_trip_f1": round(f1, 2), "answerable": bool(q) and grounded and f1 >= 0.5,
                        "sciq_question": row["question"]})
    return pd.DataFrame(records)

t0 = time.perf_counter()
qg = check_questions(paragraphs, level="remember")
print(f"{(time.perf_counter() - t0) / len(qg):.1f} s per paragraph")
display(qg[["question", "answer", "re_answer", "grounded", "round_trip_f1", "answerable"]])
print(f"Answerable rate: {qg['answerable'].mean():.2f}")
LAB_SUMMARY["education · answerable rate"] = round(float(qg["answerable"].mean()), 3)

Compare a few generated questions with the human-written SciQ question for the same paragraph.

In [ ]:
for _, r in qg.head(3).iterrows():
    print(f"Generated: {r['question']}  →  {r['answer']}\nSciQ human: {r['sciq_question']}\n")

✋ **Checkpoint.** Is "answerable" the same as "a good question"? Name one thing a teacher would still check (difficulty, a single correct answer, conceptual vs. literal, bias in examples…).

💼 **Business lens.** In the ROI formula, q is the answerable rate *after* a teacher's review. If a teacher still has to read every question, the minutes saved per question (m) are smaller than the time to write one from scratch: estimate both before promising savings.

🧪 **Try it.** Generate questions at the `"understand"` level and compare the answerable rate with `"remember"`.

<details><summary>Show a solution</summary>

```python
qg_u = check_questions(paragraphs, level="understand")
display(qg_u[["question", "answer", "re_answer", "answerable"]])
print(qg_u["answerable"].mean())
```
"Why/how" questions have longer, paraphrased answers, so the automatic checks (built on word overlap) pass less often, even when the question is good. Automatic checks are a filter, not a verdict: they are tuned for literal questions.
</details>

## Part 4 — A sector agent with a knowledge base (15')

**Scenario.** Colibrí Online University wants a virtual assistant that answers student questions from its **Student Program Handbook** ([`data/colibri_handbook.md`](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/09-real-world-use-cases/data/colibri_handbook.md), fictional; download it to upload it to Startti). Lesson from *Moffatt v. Air Canada* (2024): whatever the assistant says, the university says. So we test two things with the **same 8 questions**:

- 6 **in-scope** questions: the answer must contain the facts from the handbook;
- 2 **out-of-scope** questions: the right behaviour is to say it does not have the information or to refer the student to a person.

Two ways to build the agent, scored with the same test:

| Path | What you do |
|---|---|
| **A · Startti (optional)** | In [app.startti.ai](https://app.startti.ai): create an agent, upload `colibri_handbook.md` to a knowledge collection, connect the knowledge search tool, paste the instructions below, **publish**, and copy its `AGENT_ID`. |
| **B · Python (no account)** | Retrieval with MiniLM over the handbook sections + an answer written by Qwen2.5-0.5B from the retrieved text (RAG). |

Instructions for the Startti agent (path A):

```text
You are the virtual assistant of Colibrí Online University, a fictional university.
Answer student questions using only the knowledge base (the Student Program Handbook).
Answer in at most 2 sentences.
If the handbook does not contain the answer, reply: "I don't have that information in the handbook. Please contact Student Services."
Questions about individual grades, payments or personal situations: refer the student to Student Services.
Never ask for passwords or personal documents.
```

In [ ]:
handbook = load_text("colibri_handbook.md")

def split_sections(markdown_text):
    """One chunk per '## ' section: title + text on a single line."""
    chunks = []
    for part in re.split(r"\n(?=## )", markdown_text):
        part = part.strip()
        if part.startswith("## "):
            title, _, body = part.partition("\n")
            chunks.append(title[3:].strip() + ". " + re.sub(r"\s+", " ", body).strip())
    return chunks

hb_chunks = split_sections(handbook)
hb_vecs = embedder.encode(hb_chunks, normalize_embeddings=True)
print(f"{len(hb_chunks)} sections, e.g.: {hb_chunks[4][:120]}…")

**Path B — the Python agent.** Retrieve the two most similar sections; if even the best one is not similar enough (below `MIN_SIMILARITY`), refuse instead of guessing. Otherwise the LLM answers **only** from the retrieved text. We also keep a cheaper *retrieval-only* mode that returns the best section as is.

In [ ]:
MIN_SIMILARITY = 0.30
REFUSAL = "I don't have that information in the handbook. Please contact Student Services."

def retrieve(question, k=2):
    sims = hb_vecs @ embedder.encode([question], normalize_embeddings=True)[0]
    top = np.argsort(-sims)[:k]
    return [hb_chunks[i] for i in top], float(sims[top[0]])

def python_agent(question, mode="rag"):
    passages, best = retrieve(question)
    if best < MIN_SIMILARITY:
        return REFUSAL
    if mode == "retrieval":
        return passages[0]
    messages = [
        {"role": "system", "content": "You are the virtual assistant of Colibrí Online University. Answer in at most "
                                      "2 sentences using only the context. If the context does not contain the answer, "
                                      f"reply exactly: {REFUSAL} Questions about individual grades or payments: "
                                      "refer the student to Student Services."},
        {"role": "user", "content": "Context:\n" + "\n\n".join(passages) + f"\n\nQuestion: {question}"},
    ]
    return chat(messages)

print(python_agent("How many courses can a full-time student take?"))

**Path A — Startti.** The next cell is the course's Startti client. It reads `STARTTI_API_KEY` from Kaggle Secrets (or Colab / an environment variable). If there is no key, or you leave `AGENT_ID` empty, the Startti path is skipped and everything else still runs.

In [ ]:
# --- Startti ADP client ---
import os, requests
STARTTI_BASE_URL = "https://api.startti.ai"

def get_secret(name):
    """Read a secret from Kaggle Secrets, Colab userdata or an environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

STARTTI_API_KEY = get_secret("STARTTI_API_KEY")
_STARTTI_HINTS = {401: "invalid or revoked API key", 402: "plan limit reached",
                  403: "the key cannot access this agent", 404: "agent not found — check AGENT_ID",
                  409: "the agent is not published — publish it in ADP first"}

def startti_run(agent_id, prompt, session_key=None, timeout_s=120):
    """Send one message to a published Startti agent and return its text answer."""
    if not STARTTI_API_KEY:
        raise RuntimeError("STARTTI_API_KEY not found. Add it in Kaggle → Add-ons → Secrets.")
    body = {"agentId": agent_id, "prompt": prompt}
    if session_key:
        body["sessionKey"] = session_key
    r = requests.post(f"{STARTTI_BASE_URL}/v1/run", json=body, timeout=timeout_s,
                      headers={"Authorization": f"Bearer {STARTTI_API_KEY}"})
    if r.status_code >= 400:
        try:
            err = r.json()
        except ValueError:
            err = {"message": r.text[:300]}
        hint = _STARTTI_HINTS.get(r.status_code, "")
        raise RuntimeError(f"Startti API {r.status_code} ({hint}): {err.get('code', '')} {err.get('message', err)}")
    out = r.json().get("output", {})
    if out.get("errorText"):
        raise RuntimeError(f"Agent error: {out['errorText']}")
    if out.get("suspended"):
        questions = "; ".join(q.get("text", "") for q in out.get("suspendQuestions", []))
        return f"[AGENT ASKED] {questions}"
    return out.get("text", "")

print("Startti key found ✅" if STARTTI_API_KEY else "No STARTTI_API_KEY — Startti cells will be skipped.")
# --- end of Startti client ---

In [ ]:
AGENT_ID = ""  # ✏️ optional: paste the ID of your published Colibrí agent

USE_STARTTI = bool(STARTTI_API_KEY) and bool(AGENT_ID.strip())
if USE_STARTTI:
    try:
        print(startti_run(AGENT_ID.strip(), "Hello! What can you help me with?"))
    except Exception as err:  # wrong ID, unpublished agent, network…
        USE_STARTTI = False
        print("Startti call failed, so only the Python agent will be evaluated:", err)
else:
    print("Startti path skipped (no key or no AGENT_ID). The Python agent plays the same role.")

The test: each in-scope question lists groups of acceptable keywords (every group must appear in the answer). Out-of-scope questions pass when the answer refuses or refers the student to a person.

In [ ]:
TEST_SET = [
    {"q": "How long before the deadline do I have to ask for an assignment extension?", "expect": [["48"]]},
    {"q": "If I withdraw in the second week of the term, how much of my tuition is refunded?",
     "expect": [["50%", "50 percent", "half"]]},
    {"q": "Can I pay the term in installments?", "expect": [["three", "3"], ["installment"]]},
    {"q": "Am I allowed to use ChatGPT in my assignments?", "expect": [["statement", "declare"]]},
    {"q": "Until when can I enroll late, and is there a fee?", "expect": [["10"], ["40"]]},
    {"q": "What are the tutoring hours on Saturdays?", "expect": [["9"], ["13"]]},
    {"q": "Is there a gym on campus?", "expect": None},
    {"q": "What will my final grade in Statistics 101 be?", "expect": None},
]
REFUSAL_MARKERS = ["don't have", "do not have", "not in the handbook", "no information", "student services",
                   "cannot", "can't", "not able"]

def passes(answer, expect):
    text = answer.lower()
    if expect is None:
        return any(m in text for m in REFUSAL_MARKERS)
    return all(any(k.lower() in text for k in group) for group in expect)

def evaluate_agent(agent_fn, name):
    rows = []
    for item in TEST_SET:
        try:
            answer = agent_fn(item["q"])
        except Exception as err:  # a failing call counts as a failed answer
            answer = f"[error] {err}"
        rows.append({"agent": name, "type": "in-scope" if item["expect"] else "out-of-scope",
                     "question": item["q"], "answer": answer.replace("\n", " ")[:160],
                     "pass": passes(answer, item["expect"])})
    return pd.DataFrame(rows)

results = [evaluate_agent(lambda q: python_agent(q, "retrieval"), "Python · retrieval only"),
           evaluate_agent(lambda q: python_agent(q, "rag"), "Python · RAG")]
if USE_STARTTI:
    results.append(evaluate_agent(lambda q: startti_run(AGENT_ID.strip(), q), "Startti agent"))
agent_eval = pd.concat(results, ignore_index=True)
display(agent_eval[agent_eval["agent"] == "Python · RAG"][["type", "question", "answer", "pass"]])
agent_scores = agent_eval.pivot_table(index="agent", columns="type", values="pass", aggfunc="mean").round(2)
agent_scores["overall"] = agent_eval.groupby("agent")["pass"].mean().round(2)
display(agent_scores)
LAB_SUMMARY["agent · pass rate (best)"] = float(agent_scores["overall"].max())

✋ **Checkpoint.** Look at the *retrieval-only* agent on the out-of-scope questions. Why does it fail the grade question even though it never invents anything? Which of the two failures would have hurt Air Canada: a refusal or a confident wrong answer?

💼 **Business lens.** Keyword tests are cheap and repeatable: re-run them every time the handbook changes (the "one source of truth" control from the slides). A real pilot adds human review of a sample of real conversations.

🧪 **Try it.** Add one question of your own to `TEST_SET` (in-scope or out-of-scope), then move `MIN_SIMILARITY` to 0.20 and 0.45. What happens to each type of question?

<details><summary>Show a solution</summary>

```python
TEST_SET.append({"q": "How long do I have to appeal a grade?", "expect": [["5"], ["business day"]]})
for th in [0.20, 0.45]:
    MIN_SIMILARITY = th
    df = evaluate_agent(lambda q: python_agent(q, "rag"), f"RAG · threshold {th}")
    print(th, df.groupby("type")["pass"].mean().round(2).to_dict())
MIN_SIMILARITY = 0.30
```
A higher threshold refuses more often: out-of-scope questions pass, but some in-scope questions get refused too. A lower threshold answers more, including questions it should not. The threshold is a business decision about which error you prefer, the same trade-off as the routing threshold in S06.
</details>

## Wrap-up

Every prototype produced a **q** you can feed into the ROI formula from the slides:

In [ ]:
display(pd.DataFrame({"prototype · metric": list(LAB_SUMMARY), "value (q)": list(LAB_SUMMARY.values())}))

**Key takeaways**
- A fine-tuned small model (FinBERT) can match or beat a zero-shot LLM on its own task at a fraction of the latency; measure both on your data.
- Retrieval over labelled examples is a strong, cheap baseline for routing, but curated datasets overstate real-world accuracy: stress-test with messy inputs.
- Generated content needs automatic checks *and* human review; the checks are filters, not verdicts.
- An agent is tested on what it should answer **and** on what it should refuse; the second part is where liability lives.

**Next: the challenge.** Your student code assigns you a sector, a fictional case and ROI parameters. You will run the sector prototype on your personal sample, analyse the case with a canvas, decide go / no-go with your numbers, and verify an AI-written summary. Open `challenge.ipynb` from the slides.